# 2026 WCRC 학생용 템플릿 (Pinky pro)

이 노트북은 **전체 골격(프레임워크)** 만 제공합니다.
`target_list` / `after_track_list` 에는 형식을 보여주는 **예시 2개**만 들어 있고,
실제 맵에 맞는 좌표와 동작은 **여러분이 직접 측정하고 설계**해서 채워야 합니다.

## ★ 학생이 수정하는 곳은 딱 4곳!

| 번호 | 무엇을 | 어디서 (셀) | 어떻게 |
|---|---|---|---|
| **[학생 수정 ①]** | `my_ip` | 3번 셀 | PC에서 `ipconfig` 로 확인한 IP 입력 |
| **[학생 수정 ②]** | `MOVE_FORWARD_PER_ONE` | 3번 셀 | 맨 아래 **부록 셀**로 직접 측정한 값 입력 |
| **[학생 수정 ③]** | `target_list` | 4번 셀 | 주행 순서대로 마커 id 와 x, z 좌표 작성 |
| **[학생 수정 ④]** | `after_track_list` | 5번 셀 | 각 마커 도착 후 동작 설계 |

**이 4곳을 제외한 나머지 셀(6번~14번)은 수정하지 않습니다.** 코드를 읽으면서 전체 동작 흐름을 이해하는 것이 목표입니다.
(`학생 수정` 으로 검색(Ctrl+F)하면 수정할 곳을 바로 찾을 수 있습니다)

**실행 전 준비 (순서 중요)**
1. PC에서 `run.bat` 으로 Flask 서버를 먼저 실행하고, 브라우저에서 `.pt` 모델 파일을 드래그 앤 드롭으로 등록한다
2. 아래 사용자 설정 영역의 `my_ip` 에 PC의 IP 주소를 입력한다 (Windows cmd 창에서 `ipconfig` 로 확인)
3. 이 노트북의 셀을 **위에서부터 순서대로** 실행한다

> ⚠️ "메인 실행 루프" 셀을 실행하면 로봇이 실제로 주행합니다. 그 전 셀까지만 먼저 실행해 준비 상태를 확인하세요.


## 1. 라이브러리 임포트 (수정 ×)

In [ ]:
import cv2
import time
import requests
from enum import Enum
from pinkylib import Motor
from pinkylib import Buzzer
from pinkylib import Camera
from pinkylib.yolo import Yolo
from collections import Counter
from pinky_lcd.pinky_lcd import LCD
from PIL import Image, ImageSequence, ImageDraw, ImageFont

## 2. 상수 정의 (수정 ×)

`after_track_list` 에서 사용하는 **동작 코드**가 여기에 정의되어 있습니다.
숫자 자체에 의미는 없고, 이름으로 어떤 동작인지 구분하기 위한 값입니다.

> 단, `MOTOR_SPEED` / `SEARCH_MOTOR_SPEED` / `TURN_..._TIME` 은 로봇 상태에 따라 **선택적으로** 조정할 수 있습니다. (먼저 기본값으로 테스트해 본 뒤 필요할 때만!)

In [ ]:
LEFT = 0
RIGHT = 1
FORWARD = 2

# after_track_list 에서 사용하는 동작 코드
GO_STRAIGHT = 100         # 전진
MOVE_RIGHT = 101          # 우회전
MOVE_LEFT = 102           # 좌회전
GO_BACKWARD = 103         # 후진
APPLE_COUNT_ACTION = 104  # 사과 개수 세기 (flask 서버로 이미지 전송, 개수 응답받기)
APPLE_DISPLAY = 105       # LCD에 사과 개수 표시하기
CROSS_WALK_WAIT = 106     # 잠시 대기 (예: 횡단보도)
DEFAULT = 999             # 시간 옵션이 필요 없는 동작에 사용


MOTOR_SPEED = 90
SEARCH_MOTOR_SPEED = 65

SEARCH_TURN_TIME = 0.05
MATCH_TURN_TIME = 0.05
MATCH_FORWARD_TIME = 0.4

SLEEP_TIME_AFTER_MOVE = 0.15
MOTOR_BIG_STEP_FORWARD = 1
TURN_HALF_TIME = 2              # 모터 스피드에 따라 변경 필요할 수 있음
TURN_QUATER_TIME = 1            # 모터 스피드에 따라 변경 필요할 수 있음
STRAIGHT_TO_MAIN_ROAD_TIME = 3  # 모터 스피드에 따라 변경 필요할 수 있음


# 사과 갯수가 담기는 전역 변수
# (빨간 사과, 초록 사과를 구분하지 않고 읽은 총 갯수)
total_apple_count = 0

## 3. 사용자 설정 영역 — ✏️ [학생 수정 ①] [학생 수정 ②]

| 변수 | 의미 | 수정 |
|---|---|---|
| `SEARCH_COUNT` | 아루코 마커 찾을 횟수 | 기본값 사용 |
| `APPLE_CHECK_COUNT` | 사과 이미지를 flask 서버에 보낼 횟수 | 기본값 사용 |
| `my_ip` | Flask 서버가 실행 중인 PC의 IP 주소 | ✏️ **[학생 수정 ①]** |
| `MOVE_FORWARD_PER_ONE` | `time.sleep(1)` 초 동안 전진했을 때의 아루코 마커 z축 변화 값 | ✏️ **[학생 수정 ②]** |

- **[학생 수정 ①]** `my_ip` 는 Windows cmd 창에서 `ipconfig` 로 확인한 주소를 입력하세요.
- **[학생 수정 ②]** `MOVE_FORWARD_PER_ONE` 은 로봇마다 다릅니다. 맨 아래 **부록 셀**로 직접 측정해서 수정하세요. (음수면 절댓값 사용)

In [ ]:
SEARCH_COUNT = 5          # 아루코 마커 찾을 횟수 (기본값 사용)
APPLE_CHECK_COUNT = 3     # 사과 이미지 flask 서버에 보낼 횟수 (기본값 사용)

# ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼
# ★ [학생 수정 ①] my_ip — Flask 서버 PC의 IP 주소
# ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼

my_ip = "192.168.x.x"     # ← ★ 본인 PC의 IP 주소로 변경하세요 (cmd -> ipconfig)

# ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼
# ★ [학생 수정 ②] MOVE_FORWARD_PER_ONE — 직접 측정!
# ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼

MOVE_FORWARD_PER_ONE = 43.792   # ← ★ 본인 로봇으로 측정한 값으로 변경하세요 (부록 셀 참고)

## 4. 목표 아루코 마커 리스트 — ✏️ [학생 수정 ③]

로봇이 **주행할 순서대로** 마커를 적는 리스트입니다.

`pose : [x, z, direction_to_search]`
- `x` : 마커 앞에 정렬을 마쳤을 때의 목표 x 값 (카메라 기준 좌우 위치)
- `z` : 마커 앞에 멈출 때의 목표 z 값 (마커까지의 거리)
- `direction_to_search` : 마커 탐색을 시작할 회전 방향
  - `LEFT` : 로봇 기준 **왼쪽부터** 회전하며 탐색
  - `RIGHT` : 로봇 기준 **오른쪽부터** 회전하며 탐색

**측정 방법**: 맵에서 로봇을 "마커 앞에 세우고 싶은 위치"에 놓고, 부록 셀의 인식 코드로 그때의 `x`, `z` 값을 읽어 기록하세요.

> 아래 2개는 형식을 보여주는 **예시**입니다. 실제 맵 값으로 바꾸고, 주행 순서대로 항목을 추가하세요.

In [ ]:
# ★ [학생 수정 ③] 아루코마커 주행 순서 id 리스트
target_list = [
    {"id": 1, "pose": [12.56, 55, RIGHT]},   # 예시 1 (★ 실제 맵 값으로 교체)
    {"id": 3, "pose": [6, 75, RIGHT]},       # 예시 2 (★ 실제 맵 값으로 교체)
    # {"id": ?, "pose": [?, ?, ?]},          # <- ★ 여기서부터 직접 추가
]

## 5. 마커 도착 후 동작 리스트 — ✏️ [학생 수정 ④]

각 마커에 **도착한 다음** 수행할 동작들입니다.
`target_list` 와 **같은 순서, 같은 개수**로 작성해야 합니다. (i번째 마커 도착 → i번째 `actions` 실행)

| 동작 | 형식 | 설명 |
|---|---|---|
| 전진 | `(GO_STRAIGHT, 초)` | duration_time으로 얼마나 전진할지 조절 |
| 후진 | `(GO_BACKWARD, 초)` | duration_time으로 얼마나 후진할지 조절 |
| 좌회전 | `(MOVE_LEFT, 초)` | duration_time으로 얼마나 회전할지 조절 |
| 우회전 | `(MOVE_RIGHT, 초)` | duration_time으로 얼마나 회전할지 조절 |
| 사과 개수 세기 | `(APPLE_COUNT_ACTION, DEFAULT)` | flask 서버로 이미지 전송하고 detect 된 개수 응답받기 |
| 사과 개수 표시 | `(APPLE_DISPLAY, DEFAULT)` | LCD에 누적 사과 개수 표시하기 |
| 잠시 대기 | `(CROSS_WALK_WAIT, DEFAULT)` | 0.5초 대기 |

아무 동작이 없을 경우 빈 칸으로 입력해줍니다: `{"id": 0, "actions": []}`

In [ ]:
# ★ [학생 수정 ④] 마커 도착 후 수행할 동작 리스트
after_track_list = [
    {"id": 1, "actions": [(MOVE_RIGHT, 0.4), (GO_STRAIGHT, 0.3)]},                    # 예시 1: 주행 동작만 (★ 직접 설계)
    {"id": 3, "actions": [(APPLE_COUNT_ACTION, DEFAULT), (APPLE_DISPLAY, DEFAULT)]},  # 예시 2: 사과 세기 + LCD 표시 (★ 직접 설계)
    # {"id": ?, "actions": []},   # <- ★ 여기서부터 직접 추가
]

## 6. 하드웨어 초기화 (수정 ×)

> ★ **학생 수정 구역 끝!** 여기(6번)부터 마지막 셀까지는 **수정하지 않는** 영역입니다.

버저, 모터, 카메라, LCD 를 켭니다. **이 셀은 노트북을 켠 뒤 한 번만 실행**하세요.
(두 번 실행해서 에러가 나면 커널을 재시작한 뒤 처음부터 다시 실행합니다)

In [ ]:
SEARCH_RETURN_TIME = SEARCH_TURN_TIME * SEARCH_COUNT

pinky_buzzer = Buzzer()
pinky_motor = Motor()
pinky_cam = Camera()
pinky_lcd = LCD()

pinky_cam.set_calibration()

pinky_cam.start()
pinky_motor.enable_motor()
pinky_buzzer.buzzer_start()

## 7. Flask 서버 통신 — 사과 개수 받아오기 (수정 ×)

로봇이 리소스가 부족하기 때문에, **object detection(YOLO 추론)은 PC의 Flask 서버에서** 수행합니다.

흐름: 카메라 프레임 → JPEG 인코딩 → `POST /predict` 로 전송 → 서버가 YOLO 추론 → `detected_count` 응답

In [ ]:
def get_server_url(ip):
    return f"http://{ip}:5000/predict"

def send_image_and_get_count(image_input):  # 이미지를 PC(Flask)로 전송하고 감지된 사물의 총 개수(count)를 받아옵니다.
    try:
        # 1) 파일 경로(문자열)인 경우
        if isinstance(image_input, str):
            with open(image_input, 'rb') as f:
                files = {'image': f}
                response = requests.post(SERVER_URL, files=files)

        # 2) Picamera2 / OpenCV frame (Numpy Array)인 경우
        else:
            # 메모리 상에서 즉시 JPEG 바이너리로 인코딩 (파일 저장 없이 고속 처리)
            success, img_encoded = cv2.imencode('.jpg', image_input)
            if not success:
                print(" image encoding failed")
                return None

            files = {'image': ('robot_frame.jpg', img_encoded.tobytes(), 'image/jpeg')}
            response = requests.post(SERVER_URL, files=files)

        # 응답 처리
        if response.status_code == 200:
            res_data = response.json()

            # [디버그 출력] 서버가 실제로 보내온 데이터 전체를 확인합니다.
            print("Server respose raw data:", res_data)

            # 서버에서 'detected_count' 키값을 가져옴
            if 'detected_count' in res_data:
                count = res_data['detected_count']
            else:
                print("warning:'detected_count' key is missing in Server response")
                count = 0

            print(f"detected count: {count} (file name: {res_data.get('saved_filename')})")
            return count

        elif response.status_code == 400:
            err_msg = response.json().get('message', '요청 에러')
            print(f" Error : {err_msg}")
            return None

        else:
            print(f" Server status error (code {response.status_code}):", response.text)
            return None

    except Exception as e:
        print(f"Communication exception :", e)
        return None

## 8. LCD 표시 & 사과 카운트 (수정 ×)

- `display_apple_count()` : 검은 배경 이미지를 만들어 `Total apple count : N` 텍스트를 가운데에 그린 뒤 LCD에 출력
- `predict_apple_count()` : `APPLE_CHECK_COUNT` 번 촬영해 서버로 보내고, 응답 중 **가장 큰 값**을 채택 (한두 장 인식이 흔들려도 안정적으로 세기 위함)

In [ ]:
def display_apple_count(apple_count):
    global total_apple_count
    img_width, img_height = 320, 240
    background_color = (0, 0, 0)

    img = Image.new('RGB', (img_width, img_height), color=background_color)
    draw = ImageDraw.Draw(img)

    text_color = (255, 255, 255)
    text = f"Total apple count : {total_apple_count}"

    try:
        font = ImageFont.truetype("NanumGothic.ttf", 30)
    except:
        font = ImageFont.load_default()

    bbox = draw.textbbox((0, 0), text, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]
    x = (img_width - text_width) // 2
    y = (img_height - text_height) // 2
    draw.text((x, y), text, fill=text_color, font=font)

    pinky_lcd.img_show(img)


def predict_apple_count():
    # APPLE_CHECK_COUNT 번 촬영해서 서버로 보내고, 그중 가장 큰 값을 사용합니다.
    temp_count = 0
    for i in range(APPLE_CHECK_COUNT):
        frame = pinky_cam.get_frame()
        count = send_image_and_get_count(frame)
        if count is not None:
            if count > temp_count:
                temp_count = count
    return temp_count

## 9. 아루코 마커 인식 / 탐색 (수정 ×)

- `detect_target_aruco()` : 프레임 1장에서 마커를 인식하고, **목표 id와 같을 때만** 성공으로 처리
- `find_aruco_with_try_count()` : 지정 방향으로 조금씩 회전하며 `try_count` 번 탐색, 실패 시 원래 방향으로 복귀
- `find_aruco()` : 먼저 지정한 방향으로 탐색하고, 실패하면 **반대 방향으로 한 번 더** 탐색

In [ ]:
def detect_target_aruco(aruco_num):
    frame = pinky_cam.get_frame()
    output_frame, pose = pinky_cam.detect_aruco(frame, marker_size=0.1)

    if pose is None:
        print("None is detected")
        return False, None
    elif pose[0][0] != aruco_num:
        print("Target not detected")
        print("id: ", str(pose[0][0]), "x: ", str(pose[0][1]), "y: ", str(pose[0][2]), "z: ", str(pose[0][3]))
        return False, None
    else:
        print("Target detected")
        print("id: ", str(pose[0][0]), "x: ", str(pose[0][1]), "y: ", str(pose[0][2]), "z: ", str(pose[0][3]))
        return True, pose


# direction : 0(Left), 1(Right)
def find_aruco_with_try_count(aruco_num, direction, try_count):
    if try_count != 0:
        for i in range(try_count):
            success, pose = detect_target_aruco(aruco_num)
            time.sleep(0.3)
            if success:
                print("find_aruco, detected")
                return True, pose
            else:
                if direction == LEFT:
                    move_left(SEARCH_TURN_TIME, SEARCH_MOTOR_SPEED)
                    print("left, not detected")
                else:
                    move_right(SEARCH_TURN_TIME, SEARCH_MOTOR_SPEED)
                    print("right, not detected")
                print("i", i)
                time.sleep(SLEEP_TIME_AFTER_MOVE)
        # try_count 번 다 돌아도 못 찾으면 원래 방향으로 복귀
        if direction == LEFT:
            move_right(SEARCH_RETURN_TIME)
        else:
            move_left(SEARCH_RETURN_TIME)
    return False, None


def find_aruco_forever(aruco_num, direction):
    while True:
        success, pose = detect_target_aruco(aruco_num)
        time.sleep(0.3)
        if success:
            print("no limit, detected")
            return True, pose
        else:
            if direction == LEFT:
                move_left(SEARCH_TURN_TIME, SEARCH_MOTOR_SPEED)
                print("left, no limit, not detected")
            else:
                move_right(SEARCH_TURN_TIME, SEARCH_MOTOR_SPEED)
                print("right, no limit, not detected")
            time.sleep(SLEEP_TIME_AFTER_MOVE)


def find_aruco(aruco_num, direction, try_count):
    # 지정한 방향으로 try_count 번 탐색하고, 실패하면 반대 방향으로 한 번 더 탐색합니다.
    result, pose = find_aruco_with_try_count(aruco_num, direction, try_count)
    final_result = 0
    final_pose = None

    if (result != 1):
        if direction == LEFT:
            time.sleep(SLEEP_TIME_AFTER_MOVE)
            final_result, final_pose = find_aruco_with_try_count(aruco_num, RIGHT, try_count)
        elif direction == RIGHT:
            time.sleep(SLEEP_TIME_AFTER_MOVE)
            final_result, final_pose = find_aruco_with_try_count(aruco_num, LEFT, try_count)
    else:
        return result, pose

    return final_result, final_pose

## 10. 각도/거리 정렬 & 마커 추적 (주행의 핵심 · 수정 ×)

`track_target_aruco_marker()` 가 **마커 하나에 도착하는 전 과정**입니다:

1. **탐색** — `find_aruco()` 로 목표 마커를 찾는다
2. **각도(x) 정렬** — 마커의 x 값이 목표 `x ± 15` 에 들어올 때까지 좌/우로 조금씩 회전
3. **거리(z) 접근** — 마커의 z 값이 목표 `z ± 5` 에 들어올 때까지 전진
   - 남은 거리를 `MOVE_FORWARD_PER_ONE` 으로 나눠 **전진 시간**으로 환산 → `MOVE_FORWARD_PER_ONE` 측정이 정확해야 하는 이유!

In [ ]:
def check_angle(aruco_num, target, allow_range=15):
    success, pose = detect_target_aruco(aruco_num)
    if success:
        cur_pose_x = pose[0][1]
        print("current_pose_x", cur_pose_x, "target_pose_x", target)
        if (target - allow_range < cur_pose_x) and (cur_pose_x < target + allow_range):
            return True, 0
        elif cur_pose_x < target - allow_range:
            return False, LEFT
        elif target + allow_range < cur_pose_x:
            return False, RIGHT
    else:
        print("check_angle, not detected")
        return False, None


def check_distance(aruco_num, target, allow_range=5):
    success, pose = detect_target_aruco(aruco_num)
    if success:
        cur_pose_z = pose[0][3]
        print("cur_pose_z", cur_pose_z, "target_pose_z", target)
        if (target - allow_range <= cur_pose_z) and (cur_pose_z <= target + allow_range):
            return True, None
        elif target + allow_range < cur_pose_z:
            # 아직 멀다 -> 남은 거리를 시간으로 환산해서 전진
            temp_time = abs(target - cur_pose_z) / MOVE_FORWARD_PER_ONE
            print("temp_time", temp_time)
            return False, temp_time
        elif target - allow_range > cur_pose_z:
            print("check_distance, over the target")
            return True, None
    else:
        print("check_distance, not detected")
        return False, None


def track_target_aruco_marker(aruco_num, target_pose, try_count=0):
    # 1) 마커 탐색 -> 2) 각도(x) 정렬 -> 3) 거리(z) 접근
    target_x, target_z, target_direction = target_pose

    aruco_success, pose = find_aruco(aruco_num, target_direction, try_count)
    if aruco_success:

        final_result = False

        # 각도(x) 맞추기
        while not final_result:
            final_result, angle_direction = check_angle(aruco_num, target_x)
            print("check_angle : direction  ", angle_direction)
            if not final_result:
                print("angle not ready")
                if angle_direction == LEFT:
                    move_left(MATCH_TURN_TIME)
                    time.sleep(SLEEP_TIME_AFTER_MOVE)
                else:
                    move_right(MATCH_TURN_TIME)
                    time.sleep(SLEEP_TIME_AFTER_MOVE)
        print("angle success")

        final_distance_result = False
        not_detected_count = 0

        # 거리(z) 맞추기
        while not final_distance_result:
            final_distance_result, distance_direction = check_distance(aruco_num, target_z)
            if not final_distance_result:
                print("distance not ready")
                if distance_direction is not None:
                    move_forward(distance_direction)
                    time.sleep(SLEEP_TIME_AFTER_MOVE)
                else:
                    not_detected_count += 1
                    print("distance detection failed")
                    if not_detected_count >= 3:
                        not_detected_count = 0
                        break
        return True

    else:
        print("track_target_aruco_marker find_aruco_failed")
        return False

## 11. 모터 동작 함수 (수정 ×)

모두 같은 패턴입니다: **모터 켜기 → `duration_time` 만큼 기다리기 → 정지**

In [ ]:
def move_forward(duration_time, motor_speed=MOTOR_SPEED):
    pinky_motor.move(motor_speed, motor_speed)
    time.sleep(duration_time)
    pinky_motor.move(0, 0)

def move_backward(duration_time, motor_speed=MOTOR_SPEED):
    pinky_motor.move(-motor_speed, -motor_speed)
    time.sleep(duration_time)
    pinky_motor.move(0, 0)

def move_right(duration_time, motor_speed=MOTOR_SPEED):
    pinky_motor.move(motor_speed, -motor_speed)
    time.sleep(duration_time)
    pinky_motor.move(0, 0)

def move_left(duration_time, motor_speed=MOTOR_SPEED):
    pinky_motor.move(-motor_speed, motor_speed)
    time.sleep(duration_time)
    pinky_motor.move(0, 0)


def go_straight_to_main_road(duration_time=STRAIGHT_TO_MAIN_ROAD_TIME):
    motor_speed = MOTOR_SPEED
    move_forward(duration_time)
    time.sleep(duration_time)
    pinky_motor.move(0, 0)

## 12. 마커 도착 후 동작 실행기 (수정 ×)

`after_track_list[index]` 의 `actions` 를 **앞에서부터 하나씩** 꺼내 조건문으로 분기해 실행합니다.
새 동작을 추가하고 싶다면 ① 상수 정의 → ② 이 함수에 `elif` 분기 추가 → ③ `after_track_list` 에서 사용, 순서로 확장하면 됩니다.

In [ ]:
def after_target_do_list(index):
    global total_apple_count
    action = after_track_list[index]
    current_action = action["actions"]
    if len(current_action) == 0:
        print("nothing")
        return
    for i in range(len(current_action)):
        action_inside, option_inside = current_action[i]
        if action_inside == GO_STRAIGHT:
            move_forward(option_inside)
        elif action_inside == MOVE_RIGHT:
            move_right(option_inside)
        elif action_inside == MOVE_LEFT:
            move_left(option_inside)
        elif action_inside == GO_BACKWARD:
            move_backward(option_inside)
        elif action_inside == APPLE_DISPLAY:
            display_apple_count(total_apple_count)
        elif action_inside == APPLE_COUNT_ACTION:
            temp_apple_count = predict_apple_count()
            total_apple_count = total_apple_count + temp_apple_count
        elif action_inside == CROSS_WALK_WAIT:
            time.sleep(0.5)
        else:
            print("wrong_action")

## 13. 메인 실행 루프 (수정 ×) 🚗 ⚠️ 로봇이 실제로 주행합니다!

`target_list` 를 앞에서부터 순회하며:
1. `track_target_aruco_marker()` 로 i번째 마커까지 이동
2. 성공하면 `after_target_do_list(i)` 로 도착 후 동작 실행
3. 마커를 못 찾으면 `break` 로 전체 주행 중단

In [ ]:
SERVER_URL = get_server_url(my_ip)

assert len(target_list) == len(after_track_list), \
    "target_list 와 after_track_list 의 개수가 다릅니다! 같은 순서/개수로 맞춰주세요."

for i in range(len(target_list)):
    current_id = target_list[i]["id"]
    current_target_pose = target_list[i]["pose"]
    target_x, target_z, target_direction = current_target_pose
    result = track_target_aruco_marker(current_id, current_target_pose, SEARCH_COUNT)
    if result is not True:
        break
    after_target_do_list(i)
    print("----list num : ", i, " done -----")


time.sleep(10)

## 14. 종료 — 하드웨어 해제 (수정 ×)

주행이 끝나면 반드시 실행해서 카메라/모터/버저/LCD 를 해제하세요.
(해제하지 않고 노트북을 다시 실행하면 하드웨어 점유 에러가 날 수 있습니다)

In [ ]:
pinky_cam.close()
pinky_buzzer.close()
pinky_motor.close()
pinky_lcd.close()

---
## (부록) MOVE_FORWARD_PER_ONE 측정 코드 — 실행용 (코드 수정 ×)

**[학생 수정 ②]** 에 넣을 값을 구할 때 사용하는 셀입니다.

**사용법**
1. 위의 **메인 실행 루프(13번) 셀은 실행하지 않는다** (또는 주석 처리)
2. 6번(하드웨어 초기화)과 11번(모터 함수) 셀까지는 실행해 둔다
3. 로봇을 아루코 마커가 정면에 보이는 위치에 놓는다
4. 아래 셀의 주석(`'''`)을 해제하고 실행한다
5. 출력되는 `rate` 값을 3번 셀의 `MOVE_FORWARD_PER_ONE` 에 입력한다 (**음수로 나오면 절댓값** 사용)

> ⚠️ `detect_aruco` 의 `marker_size` 는 본 코드의 `detect_target_aruco` 와 동일한 **0.1** 을 사용해야 z 값의 스케일이 맞습니다.

In [ ]:
'''
frame = pinky_cam.get_frame()
output_frame, pose = pinky_cam.detect_aruco(frame, marker_size=0.1)
pinky_cam.display_jupyter(output_frame)
print("id: ", str(pose[0][0]), "x: ", str(pose[0][1]), "y: ", str(pose[0][2]), "z: ", str(pose[0][3]))

z_1 = pose[0][3]

move_forward(1)
time.sleep(0.5)

frame = pinky_cam.get_frame()
output_frame, pose = pinky_cam.detect_aruco(frame, marker_size=0.1)
pinky_cam.display_jupyter(output_frame)
print("id: ", str(pose[0][0]), "x: ", str(pose[0][1]), "y: ", str(pose[0][2]), "z: ", str(pose[0][3]))

z_2 = pose[0][3]

print("z_1: ", str(z_1), "z_2: ", str(z_2))

rate = z_2 - z_1

print("rate: ", rate)
'''